<img src="https://github.com/hernancontigiani/ceia_memorias_especializacion/raw/master/Figures/logoFIUBA.jpg" width="500" align="center">


# Procesamiento de lenguaje natural
## Custom embedddings con Gensim



### Objetivo
El objetivo es utilizar documentos / corpus para crear embeddings de palabras basado en ese contexto. Se utilizará canciones de bandas para generar los embeddings, es decir, que los vectores tendrán la forma en función de como esa banda haya utilizado las palabras en sus canciones.

### Consigna del desafío 2

**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado**

Recuerden que su notebook de entrega debe poder correrse de inicio a fin sin la aparición de errores.

- Crear sus propios vectores con Gensim basado en lo visto en clase con un corpus propio (revisar enlaces sugeridos en clase 2 sobre opciones de dataset)
- Elegir términos de interés y buscar términos más similares y menos similares.
- Realizar una reduccion de dimensionalidad a los embeddings, llevándolos a 2 dimensiones. Graficar los embeddings proyectados y seleccionar una cantidad de términos (variable MAX_WORDS) de forma tal que la visualización sea adecuada.
- Inspeccionar el grafico y buscar pequeños grupos de palabras que puedan formarse. Interpretarlos e intentar obtener conclusiones. En lo posible, acompañar los grupos de palabras con capturas (y pegarlas en celdas de texto)

Utilizo `uv` para la gestión de paquetes. Correr en terminal desde la carpeta del repo (suponiendo que ya se cuenta con `uv` instalado):
```bash
uv sync
uv run ipython kernel install --user --env VIRTUAL_ENV $(pwd)/.venv --name=pln1     
```

Voy a usar como corpus el libro de Albert Einstein llamado "Relativity : the Special and General Theory", el cual es un libro de divulgación de la teoría de la relatividad de Einstein, apto para un púlbico no instruido en matemática y física de manera extensiva. El ebook está tomado directamente de la página The Project Gutemberg, sin modificaciones, por lo que se tienen los preámbulos del archivo además del contenido del libro propiamente.

In [43]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import multiprocessing
from gensim.models import Word2Vec


In [44]:
df_relativity = pd.read_csv('relativity_einstein.txt', sep='/n', header=None)

/tmp/ipykernel_4003/3556228936.py:1: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  df_relativity = pd.read_csv('relativity_einstein.txt', sep='/n', header=None)


In [45]:
from tensorflow.keras.preprocessing.text import text_to_word_sequence

sentence_tokens = []
# Recorrer todas las filas y transformar las oraciones
# en una secuencia de palabras (esto podría realizarse con NLTK o spaCy también)
for _, row in df_relativity[:None].iterrows():
    sentence_tokens.append(text_to_word_sequence(row[0]))

In [46]:
from gensim.models.callbacks import CallbackAny2Vec
# Durante el entrenamiento gensim por defecto no informa el "loss" en cada época
# Sobrecargamos el callback para poder tener esta información
class callback(CallbackAny2Vec):
    """
    Callback to print loss after each epoch
    """
    def __init__(self):
        self.epoch = 0

    def on_epoch_end(self, model):
        loss = model.get_latest_training_loss()
        if self.epoch == 0:
            print('Loss after epoch {}: {}'.format(self.epoch, loss))
        else:
            print('Loss after epoch {}: {}'.format(self.epoch, loss- self.loss_previous_step))
        self.epoch += 1
        self.loss_previous_step = loss

In [47]:
# Crearmos el modelo generador de vectores
# En este caso utilizaremos la estructura modelo Skipgram
w2v_model = Word2Vec(min_count=5,    # frecuencia mínima de palabra para incluirla en el vocabulario
                     window=2,       # cant de palabras antes y desp de la predicha
                     vector_size=300,       # dimensionalidad de los vectores
                     negative=20,    # cantidad de negative samples... 0 es no se usa
                     workers=1,      # si tienen más cores pueden cambiar este valor
                     sg=1)           # modelo 0:CBOW  1:skipgram

In [48]:
# Obtener el vocabulario con los tokens
w2v_model.build_vocab(sentence_tokens)

# Cantidad de words encontradas en el corpus
print("Cantidad de words distintas en el corpus:", len(w2v_model.wv.index_to_key))

Cantidad de words distintas en el corpus: 908


In [49]:
# Entrenamos el modelo generador de vectores
# Utilizamos nuestro callback
w2v_model.train(sentence_tokens,
                 total_examples=w2v_model.corpus_count,
                 epochs=30,
                 compute_loss = True,
                 callbacks=[callback()]
                 )


Loss after epoch 0: 316834.34375
Loss after epoch 1: 221160.96875
Loss after epoch 2: 221989.625
Loss after epoch 3: 222759.875
Loss after epoch 4: 200773.3125
Loss after epoch 5: 189594.5
Loss after epoch 6: 186755.375
Loss after epoch 7: 185316.625
Loss after epoch 8: 182543.0
Loss after epoch 9: 179489.625
Loss after epoch 10: 155914.0
Loss after epoch 11: 154354.5
Loss after epoch 12: 154358.25
Loss after epoch 13: 153347.75
Loss after epoch 14: 152341.5
Loss after epoch 15: 151951.0
Loss after epoch 16: 151449.0
Loss after epoch 17: 150098.25
Loss after epoch 18: 149685.75
Loss after epoch 19: 148699.75
Loss after epoch 20: 148817.75
Loss after epoch 21: 147203.75
Loss after epoch 22: 146685.0
Loss after epoch 23: 143571.5
Loss after epoch 24: 134852.5
Loss after epoch 25: 135798.0
Loss after epoch 26: 135611.5
Loss after epoch 27: 133842.5
Loss after epoch 28: 133996.0
Loss after epoch 29: 135173.5


(593229, 1040070)

In [50]:
def get_similar_different(word):
    print(f"Palabras más similares a {word}:")
    print([vector[0] for vector in w2v_model.wv.most_similar(positive=[word], topn=10)])
    print(f"Palabras más distintas a {word}:")
    print([vector[0] for vector in w2v_model.wv.most_similar(negative=[word], topn=10)])


In [51]:
print("Relativity")
get_similar_different("relativity")

print("\nEquation")
get_similar_different("equation")

Relativity
Palabras más similares a relativity:
['introduction', 'speak', 'argument', 'favour', 'experimental', 'either', 'demands', 'moreover', 'later', 'fizeau']
Palabras más distintas a relativity:
['b', 'during', 'when', 'outside', 'beings', 'f', 'straight', 'these', 'interval', 'can']

Equation
Palabras más similares a equation:
['equations', 'xi', 'perfectly', 'replace', 'fourth', '5', 'express', 'supplementary', 'relations', 'transformation']
Palabras más distintas a equation:
['far', 'based', 'situated', 'marble', 'our', 'so', 'other', 'individual', 'slab', 'surfaces']


Analizamos 2 palabras primordiales: "relativity" y "equation". <br>
Podemos ver que alrededor de "relativity" se tienen palabras que denotan que en este documento se está presentando y explicando la teoría de la relatividad. Pero en los opuestos, es dificil ver algún sentido en las palabras presentes. <br>
Para "equation" sucede algo similar, la más similiar es su plural, y luego se tienen palabras que suelen ir acompañando a equatoion. Pero en sus opuestos no hay mucho sentido.

In [52]:
from sklearn.decomposition import IncrementalPCA
from sklearn.manifold import TSNE
import numpy as np

def reduce_dimensions(model, num_dimensions = 2 ):

    vectors = np.asarray(model.wv.vectors)
    labels = np.asarray(model.wv.index_to_key)

    tsne = TSNE(n_components=num_dimensions, random_state=0)
    vectors = tsne.fit_transform(vectors)

    return vectors, labels


In [53]:
# Graficar los embedddings en 2D
import plotly.graph_objects as go
import plotly.express as px

vecs, labels = reduce_dimensions(w2v_model)

MAX_WORDS=200
fig = px.scatter(x=vecs[:MAX_WORDS,0], y=vecs[:MAX_WORDS,1], text=labels[:MAX_WORDS])
fig.show() # esto para plotly en colab


Analicemos ahora los entornos de distintas palabras en el gráfico bidimensional.

Por ejemplo alrededor de la palabra equation tenemos: equations, transformation, obtain, according. Todas palabras que se encuentran en conjunción con equation, o equations que es su plural.
![](desafio2/equation.png)

Theory y principle son conceptos similares pero se pueden usar de forma similar, por lo que su cercanía tiene sentido: ![](desafio2/theory.png)

Se tienen conceptos que se entrelazan en la teoría de la relatividad juntos: "gravitational field", "light velocity", "mass": ![](desafio2/gravitational.png)